In [1]:
import numpy as np 
import pandas as pd 
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

### Data Preprocessing for Machine Learning

In [2]:
# Load the cleaned dataset 

df = pd.read_csv('earthquakes_cleaned.csv')
df

,mag,status,tsunami,sig,net,nst,dmin,rms,gap,magType,type,longitude,latitude,depth,reported,hour,month,day_of_week,mag_depth_ratio,is_remote
0,2.70,reviewed,0,112,ak,36.0,0.30000,0.90,146.0,ml,earthquake,-141.4430,60.0150,13.100,0,23,12,6,0.191489,False
1,4.30,reviewed,0,284,us,34.0,2.00700,0.29,188.0,mb,earthquake,145.2408,43.3059,51.870,0,23,12,6,0.081332,False
2,3.08,reviewed,0,175,nc,146.0,0.03365,0.12,16.0,mw,earthquake,-122.6045,38.3775,4.640,1,23,12,6,0.546099,False
3,4.00,reviewed,0,246,us,18.0,2.42300,0.96,93.0,mb,earthquake,110.4116,-9.4878,10.000,0,23,12,6,0.363636,False
4,4.30,reviewed,0,284,us,40.0,1.71400,0.76,62.0,mb,earthquake,-66.7998,-24.1137,198.342,0,23,12,6,0.021571,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38265,2.90,reviewed,0,129,ak,96.0,0.50000,0.90,22.0,ml,earthquake,-148.8040,62.3460,26.900,0,0,12,1,0.103943,False
38266,4.20,reviewed,0,271,us,19.0,2.01200,0.82,83.0,mb,earthquake,168.4113,-17.1006,215.635,0,0,12,1,0.019387,False
38267,2.50,reviewed,0,96,ak,18.0,0.40000,0.90,81.0,ml,earthquake,-139.8570,60.4480,1.700,0,0,12,1,0.925926,False
38268,3.10,reviewed,0,148,us,34.0,0.25900,0.34,199.0,ml,earthquake,-166.3976,53.5945,65.389,0,0,12,1,0.046694,False


**Handling Rare Categories**
- The `type`, `net` and `magType` features contained several very rare categories, So to avoid unknown-category issues during transformation, rare event types were grouped into a single category called other.

In [3]:
df['type'].value_counts()

type
earthquake                37989
mining explosion            271
explosion                     4
other event                   3
experimental explosion        2
quarry blast                  1
Name: count, dtype: int64

In [4]:
# Group rare categories in 'type'

rare_types = [
    'mining explosion',
    'explosion',
    'other event',
    'experimental explosion',
    'quarry blast'
]

df['type'] = df['type'].replace(rare_types, 'other')

In [5]:
# Group rare categories in 'net'

net_counts = df['net'].value_counts()
rare_net = net_counts[net_counts < 20].index

df['net'] = df['net'].replace(rare_net, 'other')

In [6]:
# Group rare categories in 'magType'

mag_counts = df['magType'].value_counts()
rare_mag = mag_counts[mag_counts < 20].index

df['magType'] = df['magType'].replace(rare_mag, 'other')

- Split Data into Input Features & Target Variable

In [7]:
x = df.drop('reported' , axis= 1)

y = df['reported']

In [8]:
x.head()

,mag,status,tsunami,sig,net,nst,dmin,rms,gap,magType,type,longitude,latitude,depth,hour,month,day_of_week,mag_depth_ratio,is_remote
0,2.70,reviewed,0,112,ak,36.0,0.30000,0.90,146.0,ml,earthquake,-141.4430,60.0150,13.100,23,12,6,0.191489,False
1,4.30,reviewed,0,284,us,34.0,2.00700,0.29,188.0,mb,earthquake,145.2408,43.3059,51.870,23,12,6,0.081332,False
2,3.08,reviewed,0,175,nc,146.0,0.03365,0.12,16.0,mw,earthquake,-122.6045,38.3775,4.640,23,12,6,0.546099,False
3,4.00,reviewed,0,246,us,18.0,2.42300,0.96,93.0,mb,earthquake,110.4116,-9.4878,10.000,23,12,6,0.363636,False
4,4.30,reviewed,0,284,us,40.0,1.71400,0.76,62.0,mb,earthquake,-66.7998,-24.1137,198.342,23,12,6,0.021571,False


### Numerical Pipeline

In [9]:
num_cols = x.select_dtypes(include= 'number').columns
num_cols

Index(['mag', 'tsunami', 'sig', 'nst', 'dmin', 'rms', 'gap', 'longitude',
       'latitude', 'depth', 'hour', 'month', 'day_of_week', 'mag_depth_ratio'],
      dtype='str')

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

num_pipeline = Pipeline(steps=[('Scaling' , scaler)])

num_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False


### Categorical Pipeline

In [11]:
cat_cols = x.select_dtypes(include= 'string').columns
cat_cols

Index(['status', 'net', 'magType', 'type'], dtype='str')

In [12]:
# check (n)unique values.

for col in cat_cols:
    print(f'Column :' , col)
    print(f'Number of Unique Values :' ,df[col].nunique())
   

Column : status
Number of Unique Values : 2
Column : net
Number of Unique Values : 13
Column : magType
Number of Unique Values : 8
Column : type
Number of Unique Values : 2


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop= 'first', sparse_output= False , handle_unknown='ignore')

cat_pipeline = Pipeline(steps=[('OneHotEncoder' , ohe)])

cat_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('OneHotEncoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequen

### Handling `is_remote` 

`is_remote` was saved as True/False, which models can't use directly. We 
convert it to 0/1, and since it's already binary, it needs no scaling — 
it's passed through the pipeline unchanged via `remainder='passthrough'`, 
alongside the scaled numeric columns and one-hot encoded categorical ones.

In [14]:
x['is_remote'] = x['is_remote'].astype(int)

### Assign each column to its corresponding Pipeline

In [15]:
from sklearn.compose import ColumnTransformer


preprocessing_pipeline = ColumnTransformer(transformers= [
                                                ('Numerical Pipeline', num_pipeline, num_cols),
                                                ('Categorical Pipeline', cat_pipeline, cat_cols),
                                                
] ,remainder='passthrough' )

preprocessing_pipeline

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical Pipeline', ...), ('Categorical Pipeline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``

- Split Data into Train & Test to have unseen data after CV

In [16]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

### Compare Models using only the default parameters 

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = [
    ('Logistic Regression', LogisticRegression(random_state= 42)),
    ('KNN', KNeighborsClassifier()),
    ('Gaussian NB', GaussianNB()),
    ('SVC', SVC(random_state= 42)),
    ('Decision Tree', DecisionTreeClassifier(random_state= 42)),
    ('Random Forest', RandomForestClassifier(random_state= 42)),
    ('XGBoost', XGBClassifier(random_state= 42))
]

for model in models:
    
    model_pipeline = Pipeline(steps= [('Preprocessing', preprocessing_pipeline),
                                      ('Model', model[1])])
    
    cv_result = cross_validate(estimator= model_pipeline,
                                X= x_train,
                                y= y_train,
                                scoring= 'accuracy',
                                cv= cv,
                                return_train_score= True,
                                n_jobs=-1
                                )
    
    print(model[0])
    print('Train Accuracy :', (cv_result['train_score'].mean() * 100).round(4))
    print('Test Accuracy :', (cv_result['test_score'].mean() * 100).round(4))
    print('\n')

Logistic Regression
Train Accuracy : 90.9884
Test Accuracy : 90.9655


KNN
Train Accuracy : 92.9449
Test Accuracy : 90.763


Gaussian NB
Train Accuracy : 88.2839
Test Accuracy : 88.2741


SVC
Train Accuracy : 91.4693
Test Accuracy : 91.1843


Decision Tree
Train Accuracy : 100.0
Test Accuracy : 89.7047


Random Forest
Train Accuracy : 99.9976
Test Accuracy : 92.9612


XGBoost
Train Accuracy : 98.0705
Test Accuracy : 94.3918




**Check for Imbalance Class Problem*

In [18]:
(y.value_counts(normalize=True) * 100).round(2)


reported
0    87.83
1    12.17
Name: proportion, dtype: float64

### Handling Class Imbalance with `class_weight`

**Current Issue:** Dataset is imbalanced (88% vs 12%)

**Solution:** Use `class_weight='balanced'` to give more importance to the minority class

**What to expect:**
- ↑ Recall & F1-Score for class 1

**Metrics to focus on:** Recall, F1-Score (not just Accuracy)

In [19]:
major_class = y_train.value_counts()[0]
major_class

np.int64(26890)

In [20]:
minor_class = y_train.value_counts()[1]
minor_class

np.int64(3726)

In [21]:
scale_pos_weight = round((major_class / minor_class) , 5)
scale_pos_weight

np.float64(7.21685)

- KNN & GaussianNB were removed because they don't support class_weight and weren't competitive.

In [22]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = [
    ('Logistic Regression', LogisticRegression(random_state= 42, class_weight= 'balanced')),
    ('SVC', SVC(random_state= 42 , class_weight= 'balanced')),
    ('Decision Tree', DecisionTreeClassifier(random_state= 42, class_weight = 'balanced')),
    ('Random Forest', RandomForestClassifier(random_state= 42, class_weight = 'balanced')),
    ('XGBoost', XGBClassifier(random_state= 42, scale_pos_weight = scale_pos_weight))
]

for model in models:
    
    model_pipeline = Pipeline(steps= [('Preprocessing', preprocessing_pipeline),
                                      ('Model', model[1])])
    
    cv_result = cross_validate(estimator= model_pipeline,
                                X= x_train,
                                y= y_train,
                                scoring=['precision', 'recall', 'f1'],
                                cv= cv,
                                return_train_score= True,
                                n_jobs= -1
                                )
   
    print(model[0])

    print('Train Precision :', (cv_result['train_precision'].mean() * 100).round(2))
    print('Test Precision :', (cv_result['test_precision'].mean() * 100).round(2))

    print('Train Recall :', (cv_result['train_recall'].mean() * 100).round(2))
    print('Test Recall :', (cv_result['test_recall'].mean() * 100).round(2))

    print('Train F1-Score :', (cv_result['train_f1'].mean() * 100).round(2))
    print('Test F1-Score :', (cv_result['test_f1'].mean() * 100).round(2))
    
    print('\n')

Logistic Regression
Train Precision : 38.75
Test Precision : 38.72
Train Recall : 79.78
Test Recall : 79.55
Train F1-Score : 52.16
Test F1-Score : 52.09


SVC
Train Precision : 43.11
Test Precision : 42.14
Train Recall : 86.27
Test Recall : 84.17
Train F1-Score : 57.49
Test F1-Score : 56.16


Decision Tree
Train Precision : 100.0
Test Precision : 57.65
Train Recall : 100.0
Test Recall : 54.4
Train F1-Score : 100.0
Test F1-Score : 55.95


Random Forest
Train Precision : 98.11
Test Precision : 69.53
Train Recall : 100.0
Test Recall : 70.24
Train F1-Score : 99.05
Test F1-Score : 69.87


XGBoost
Train Precision : 78.83
Test Precision : 64.06
Train Recall : 99.49
Test Recall : 81.45
Train F1-Score : 87.96
Test F1-Score : 71.71




**XGBoost model** is the best fit for this situation. 

Critical Focus: Minimizing False Negatives (FN)

- **Target Class**: `1` (Strong Earthquake)
- **Worst Case (False Negative)**: Model predicts `0` (Safe), but it's actually `1` (Danger). *This means failing to warn people.*
- **Priority Metric**: **Recall** for Class 1. 
- **Strategy**: Catching as many actual earthquakes as possible. It is better to have a few False Alarms (False Positives) than to miss a real disaster (False Negative).

In [23]:
final_model = Pipeline(steps= [('Preprocessing', preprocessing_pipeline),
                               ('Model' , XGBClassifier(random_state= 42, scale_pos_weight = scale_pos_weight))
                               ])
final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Preprocessing', ...), ('Model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Numerical Pipeline', ...), ('Categorical Pipeline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sp

### Hyperparameter Tuning


In [24]:
param_grid = {
    'Model__max_depth': [4, 5, 6],
    'Model__learning_rate': [0.05, 0.1, 0.2],
    'Model__n_estimators': [200, 300, 400],
    'Model__subsample': [0.8, 0.9, 1.0],
    'Model__colsample_bytree': [0.8, 0.9, 1.0],
    'Model__min_child_weight': [3, 5, 7],
    'Model__gamma': [0.1, 0.3, 0.5],
    'Model__reg_alpha': [0.1, 1, 5],
    'Model__reg_lambda': [1, 5, 10]
}

In [25]:
from sklearn.model_selection import RandomizedSearchCV

randomized_search = RandomizedSearchCV(
                                   estimator= final_model, 
                                   param_distributions= param_grid,
                                    n_iter= 50,
                                    cv= 5, 
                                    scoring=['precision', 'recall', 'f1'], 
                                    refit='f1',
                                    return_train_score= True,
                                    n_jobs= -1,
                                    random_state=42 
                                    )
randomized_search.fit(x_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'Model__colsample_bytree': [0.8, 0.9, ...], 'Model__gamma': [0.1, 0.3, ...], 'Model__learning_rate': [0.05, 0.1, ...], 'Model__max_depth': [4, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","['precision', 'recall', ...]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedw

In [26]:
print("Best Parameters:", randomized_search.best_params_)
print("Best F1 Score:", round(randomized_search.best_score_ * 100 , 2))

Best Parameters: {'Model__subsample': 1.0, 'Model__reg_lambda': 5, 'Model__reg_alpha': 1, 'Model__n_estimators': 400, 'Model__min_child_weight': 5, 'Model__max_depth': 5, 'Model__learning_rate': 0.2, 'Model__gamma': 0.3, 'Model__colsample_bytree': 1.0}
Best F1 Score: 73.67


- Final Evaluation on Test Data

In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report

best_model = randomized_search.best_estimator_

y_pred = best_model.predict(x_test)

print("=" * 60)
print("FINAL EVALUATION ON TEST SET (Unseen Data)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Weak (0)', 'Strong (1)']))

print("=" * 60)
print("KEY METRICS SUMMARY")
print("=" * 60)
print(f"Test Accuracy  : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Test Precision : {precision_score(y_test, y_pred) * 100:.2f}%")
print(f"Test Recall    : {recall_score(y_test, y_pred) * 100:.2f}%")
print(f"Test F1-Score  : {f1_score(y_test, y_pred) * 100:.2f}%")
print("=" * 60)

FINAL EVALUATION ON TEST SET (Unseen Data)
              precision    recall  f1-score   support

    Weak (0)       0.98      0.94      0.96      6723
  Strong (1)       0.65      0.85      0.73       931

    accuracy                           0.93      7654
   macro avg       0.81      0.89      0.85      7654
weighted avg       0.94      0.93      0.93      7654

KEY METRICS SUMMARY
Test Accuracy  : 92.57%
Test Precision : 64.88%
Test Recall    : 84.75%
Test F1-Score  : 73.50%


**Evaluate using Threshold to get a balanced score of both Recall & Precision**

In [28]:
y_proba = best_model.predict_proba(x_test)[:, 1]

thresholds = [0.3, 0.35, 0.4, 0.45, 0.5]

print(f"{'Threshold':<10}{'Precision':<12}{'Recall':<10}{'F1':<8}")
print("-" * 42)

for th in thresholds:
    y_th = (y_proba >= th).astype(int)
    print(f"{th:<10}"
          f"{precision_score(y_test, y_th)*100:<12.2f}"
          f"{recall_score(y_test, y_th)*100:<10.2f}"
          f"{f1_score(y_test, y_th)*100:<8.2f}")

Threshold Precision   Recall    F1      
------------------------------------------
0.3       52.97       90.01     66.69   
0.35      55.95       88.94     68.69   
0.4       58.71       87.22     70.18   
0.45      62.17       86.14     72.22   
0.5       64.88       84.75     73.50   





> **Note:** This section is **exploratory only** — the final model was evaluated and saved
> using the default threshold (0.5).
> It is kept here as a reference: if we ever need higher Recall in the future


### ✅ Final Model: Tuned XGBoost

The best result across all experiments (evaluated on unseen test set):

| Precision | Recall | F1-Score | Accuracy |
|-----------|--------|----------|----------|
| 64.88%    | 84.75% | **73.50%** | 92.57% |

- Beats the default model (70.88%) and the constrained tuning (65.44%).
- Train ≈ Test performance → the model generalizes well.


### Save Model

In [29]:
import joblib

joblib.dump(randomized_search.best_estimator_, 'earthquake_model_final.pkl')
print("Model saved successfully!")

Model saved successfully!


In [30]:
joblib.load('earthquake_model_final.pkl').predict(x.head(4))

array([0, 0, 1, 0])